In [32]:
import numpy as np
import pandas as pd

In [33]:
import os

json_folder = 'json_folder'
if not os.path.exists(json_folder):
    os.makedirs(json_folder)
    print(f"Created folder: {json_folder}")
else:
    print(f"Folder already exists: {json_folder}")

Folder already exists: json_folder


In [34]:
# Show all columns
pd.set_option('display.max_columns', None)

# Show all rows
pd.set_option('display.max_rows', None)

# Increase column width to see full content
pd.set_option('display.max_colwidth', None)

In [35]:
df = pd.read_csv('supplier_performance_data.csv')
df['Active_Disruptions'] = df['Active_Disruptions'].fillna('No disruption')
df['Region'] = df['Region'].fillna('Unknown')
df['Country'] = df['Country'].fillna('Unknown')
df.head()

,PO_ID,Supplier_ID,Supplier_Name,Region,Country,Product_Category,Product_SKU,Unit_Cost_USD,Lead_Time_Days,OTD_Rate_Pct,Defect_Rate_Pct,Compliance_Score,Contract_Tier,Payment_Terms_Days,MOQ_Units,Annual_Volume_Units,Risk_Level,Certifications,Last_Audit_Date,Sustainability_Score,Active_Disruptions,Alt_Supplier_ID,PO_Quarter,Units_Ordered,Units_Delivered_On_Time,Units_Rejected,PO_Value_USD
0,PO-10001,SUP-001,Orrentek Precision Mfg,APAC,China,Electronic Components,SKU-EC-1001,7.78,14,95.4,0.14,88,Tier-1,60,5943,428323,Low,ISO9001;ISO14001;RoHS,2023-08-12,77,No disruption,SUP-006,Q1-2023,9013,8598,12,70121.14
1,PO-10002,SUP-002,Zhenlong ElectroCo,APAC,China,Electronic Components,SKU-EC-1002,5.04,34,82.0,2.09,84,Tier-2,45,19856,979796,Medium,ISO9001;RoHS,2023-06-13,60,Regulatory audit pending,SUP-114,Q1-2023,69585,57059,1454,350708.40
2,PO-10003,SUP-003,Wanlong Composites Ltd,APAC,China,Raw Materials,SKU-RM-1001,10.23,22,89.9,1.35,71,Tier-2,45,5349,294502,High,ISO9001,2023-02-14,61,Audit overdue; compliance review,SUP-081,Q1-2024,18572,16696,250,189991.56
3,PO-10004,SUP-004,Yangtze Fiber Materials,APAC,China,Industrial Textiles,SKU-IT-1001,2.19,38,90.4,1.90,79,Tier-2,45,13850,1310795,Medium,OEKO-TEX;ISO9001,2023-08-27,53,Geopolitical tension flag,SUP-034,Q1-2024,19079,17247,362,41783.01
4,PO-10005,SUP-005,Huabei Circuit Systems,APAC,China,Electronic Components,SKU-EC-1003,4.46,18,98.0,0.75,94,Tier-1,60,3432,226264,Medium,ISO9001;AEC-Q100;RoHS,2023-06-25,90,Geopolitical tension flag,SUP-088,Q2-2023,13412,13143,100,59817.52


In [36]:
print("Checking for remaining NaN values in the 'df' DataFrame:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Checking for remaining NaN values in the 'df' DataFrame:
Series([], dtype: int64)


In [37]:
agg = df.groupby('Supplier_ID').agg(
    Supplier_Name=('Supplier_Name', 'first'),
    Region=('Region', 'first'),
    Country=('Country', 'first'),
    Avg_Compliance=('Compliance_Score', 'mean'),
    Avg_OTD=('OTD_Rate_Pct', 'mean'),
    Avg_Defect=('Defect_Rate_Pct', 'mean'),
    Total_Spend=('PO_Value_USD', 'sum'),
    PO_Count=('PO_ID', 'count'),
).reset_index()

agg.head()

,Supplier_ID,Supplier_Name,Region,Country,Avg_Compliance,Avg_OTD,Avg_Defect,Total_Spend,PO_Count
0,SUP-001,Orrentek Precision Mfg,APAC,China,86.277778,95.677778,0.865556,939643.31,18
1,SUP-002,Zhenlong ElectroCo,APAC,China,77.777778,84.922222,1.556111,3923690.41,18
2,SUP-003,Wanlong Composites Ltd,APAC,China,79.722222,88.000000,1.786111,6882034.93,18
3,SUP-004,Yangtze Fiber Materials,APAC,China,81.111111,87.494444,1.622222,1429755.28,18
4,SUP-005,Huabei Circuit Systems,APAC,China,90.166667,93.694444,0.943889,908603.80,18


In [38]:
def compliance_tier(comp):
    if comp >= 90: return 1
    if comp >= 75: return 2
    if comp >= 60: return 3
    return 4  # below Tier-3 floor

def otd_tier(otd):
    if otd >= 93: return 1
    if otd >= 84: return 2
    if otd >= 75: return 3
    return 4

def defect_tier(defect):  # lower is better
    if defect < 1.0: return 1
    if defect <= 2.5: return 2
    if defect <= 4.0: return 3
    return 4

def spend_tier(spend):
    if spend > 5_000_000: return 1
    if spend >= 1_000_000: return 2
    return 3

In [39]:
def classify(row):
    c = compliance_tier(row['Avg_Compliance'])
    o = otd_tier(row['Avg_OTD'])
    d = defect_tier(row['Avg_Defect'])
    s = spend_tier(row['Total_Spend'])
    worst = max(c, o, d, s)
    tier = 'Below Tier-3' if worst == 4 else f'Tier-{worst}'
    return pd.Series({'C': c, 'O': o, 'D': d, 'S': s, 'Computed_Tier': tier})

agg = pd.concat([agg, agg.apply(classify, axis=1)], axis=1)

# Sort by numeric supplier ID
agg['_num'] = agg['Supplier_ID'].str.replace('SUP-', '', regex=False).astype(int)
agg = agg.sort_values('_num').drop(columns='_num').reset_index(drop=True)
agg.head(20)

,Supplier_ID,Supplier_Name,Region,Country,Avg_Compliance,Avg_OTD,Avg_Defect,Total_Spend,PO_Count,C,O,D,S,Computed_Tier
0,SUP-001,Orrentek Precision Mfg,APAC,China,86.277778,95.677778,0.865556,939643.31,18,2,1,1,3,Tier-3
1,SUP-002,Zhenlong ElectroCo,APAC,China,77.777778,84.922222,1.556111,3923690.41,18,2,2,2,2,Tier-2
2,SUP-003,Wanlong Composites Ltd,APAC,China,79.722222,88.000000,1.786111,6882034.93,18,2,2,2,1,Tier-2
3,SUP-004,Yangtze Fiber Materials,APAC,China,81.111111,87.494444,1.622222,1429755.28,18,2,2,2,2,Tier-2
4,SUP-005,Huabei Circuit Systems,APAC,China,90.166667,93.694444,0.943889,908603.80,18,1,1,1,3,Tier-3
5,SUP-006,Dongfeng Castings Co,APAC,China,76.277778,86.622222,1.816111,2186462.26,18,2,2,2,2,Tier-2
6,SUP-007,Longhua Polymer Works,APAC,China,61.833333,71.238889,2.771111,7102925.23,18,3,4,3,1,Below Tier-3
7,SUP-008,Shengda Pack Industries,APAC,China,76.166667,88.205556,1.958333,884286.30,18,2,2,2,3,Tier-3
8,SUP-009,Tianhe Alloy Group,APAC,China,81.388889,87.027778,1.878889,8739796.90,18,2,2,2,1,Tier-2
9,SUP-010,Bohai Electronics,APAC,China,62.055556,71.261111,3.400000,3436726.31,18,3,4,3,2,Below Tier-3


In [40]:
print(agg['Computed_Tier'].value_counts())

Computed_Tier
Tier-2          61
Tier-3          34
Below Tier-3    21
Name: count, dtype: int64


In [41]:
import json

records = []
for _, r in agg.iterrows():
    # criterion/criteria that pulled the tier down (the worst band)
    worst = max(r['C'], r['O'], r['D'], r['S'])
    binding = [name for name, col in
               [('compliance', 'C'), ('OTD', 'O'),
                ('defect rate', 'D'), ('annual spend', 'S')]
               if r[col] == worst]

    text = (
        f"{r['Supplier_Name']} ({r['Supplier_ID']}) is classified as {r['Computed_Tier']} "
        f"under Section 2 of the BQBYTE Supplier Governance Policy. "
        f"Based in {r['Country']} ({r['Region']}). "
        f"Across {r['PO_Count']} purchase orders the supplier averaged a compliance score of "
        f"{r['Avg_Compliance']:.1f}, on-time delivery rate of {r['Avg_OTD']:.1f}%, and defect "
        f"rate of {r['Avg_Defect']:.2f}%, with total annual spend of ${r['Total_Spend']:,.0f}. "
        f"The tier is limited by {', '.join(binding)}."
    )

    records.append({
        "id": r['Supplier_ID'],
        "text": text,                      # this field gets embedded
        "metadata": {                      # these fields are for filtering
            "supplier_id": r['Supplier_ID'],
            "supplier_name": r['Supplier_Name'],
            "region": r['Region'],
            "country": r['Country'],
            "computed_tier": r['Computed_Tier'],
            "avg_compliance": round(r['Avg_Compliance'], 2),
            "avg_otd": round(r['Avg_OTD'], 2),
            "avg_defect": round(r['Avg_Defect'], 2),
            "total_spend_usd": round(r['Total_Spend'], 2),
            "po_count": int(r['PO_Count']),
            "binding_criteria": binding,
        },
    })

with open('./json_folder/supplier_tiers_rag.jsonl', 'w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

print(f"Saved {len(records)} records to supplier_tiers_rag.jsonl")

Saved 116 records to supplier_tiers_rag.jsonl


## **3. Service Level Aggrement by Tier**

### **3.1. On-time Delivery(OTD)**

In [42]:
# ============================================================
#  On-Time Delivery (OTD) Analysis  —  Section 3.1
# ============================================================
# Section 3.1 OTD floors by tier: Tier-1 >= 93.0%, Tier-2 >= 84.0%, Tier-3 >= 75.0%
# OTD = units delivered on time / units ordered, aggregated per supplier.

import json

# --- Aggregate OTD per supplier (reuses `df` from earlier cells) ---
otd = df.groupby('Supplier_ID').agg(
    Supplier_Name=('Supplier_Name', 'first'),
    Region=('Region', 'first'),
    Country=('Country', 'first'),
    Category=('Product_Category', 'first'),
    Contract_Tier=('Contract_Tier', 'first'),
    Reported_OTD=('OTD_Rate_Pct', 'mean'),           # avg of the CSV's per-PO OTD
    Units_Ordered=('Units_Ordered', 'sum'),
    Units_On_Time=('Units_Delivered_On_Time', 'sum'),
    PO_Count=('PO_ID', 'count'),
).reset_index()

# Recomputed OTD from raw units (the authoritative Section 3.1 figure)
otd['Computed_OTD'] = (otd['Units_On_Time'] / otd['Units_Ordered'] * 100).round(2)
otd['Reported_OTD'] = otd['Reported_OTD'].round(2)

# --- Evaluate against the supplier's assigned-tier floor ---
OTD_FLOOR = {'Tier-1': 93.0, 'Tier-2': 84.0, 'Tier-3': 75.0}
otd['OTD_Floor_Required'] = otd['Contract_Tier'].map(OTD_FLOOR)

def otd_status(r):
    floor = OTD_FLOOR.get(r['Contract_Tier'])
    if floor is None:
        return 'Unknown'
    return 'Meets SLA' if r['Computed_OTD'] >= floor else 'Below SLA'

otd['OTD_SLA_Status'] = otd.apply(otd_status, axis=1)

# Sort by numeric supplier ID
otd['_n'] = otd['Supplier_ID'].str.replace('SUP-', '', regex=False).astype(int)
otd = otd.sort_values('_n').drop(columns='_n').reset_index(drop=True)

otd.head()

,Supplier_ID,Supplier_Name,Region,Country,Category,Contract_Tier,Reported_OTD,Units_Ordered,Units_On_Time,PO_Count,Computed_OTD,OTD_Floor_Required,OTD_SLA_Status
0,SUP-001,Orrentek Precision Mfg,APAC,China,Electronic Components,Tier-1,95.68,136466,131520,18,96.38,93.0,Meets SLA
1,SUP-002,Zhenlong ElectroCo,APAC,China,Electronic Components,Tier-2,84.92,653138,562488,18,86.12,84.0,Meets SLA
2,SUP-003,Wanlong Composites Ltd,APAC,China,Raw Materials,Tier-2,88.00,536511,470803,18,87.75,84.0,Meets SLA
3,SUP-004,Yangtze Fiber Materials,APAC,China,Industrial Textiles,Tier-2,87.49,470242,416060,18,88.48,84.0,Meets SLA
4,SUP-005,Huabei Circuit Systems,APAC,China,Electronic Components,Tier-1,93.69,130131,122140,18,93.86,93.0,Meets SLA


In [43]:
# --- Save RAG-ready JSONL ---
records = []
for _, r in otd.iterrows():
    gap = (round(r['Computed_OTD'] - r['OTD_Floor_Required'], 2)
           if pd.notna(r['OTD_Floor_Required']) else None)

    text = (
        f"{r['Supplier_Name']} ({r['Supplier_ID']}) has an on-time delivery (OTD) rate of "
        f"{r['Computed_OTD']:.1f}% across {r['PO_Count']} purchase orders "
        f"({int(r['Units_On_Time']):,} of {int(r['Units_Ordered']):,} units delivered on time). "
        f"As a {r['Contract_Tier']} supplier in {r['Country']} ({r['Region']}) supplying "
        f"{r['Category']}, the Section 3.1 OTD floor is "
        f"{r['OTD_Floor_Required']:.1f}%, so the supplier {r['OTD_SLA_Status']} "
        f"(gap of {gap:+.1f} percentage points)."
    )

    records.append({
        "id": r['Supplier_ID'],
        "text": text,                          # embedded field
        "metadata": {                          # filterable fields
            "supplier_id": r['Supplier_ID'],
            "supplier_name": r['Supplier_Name'],
            "region": r['Region'],
            "country": r['Country'],
            "category": r['Category'],
            "contract_tier": r['Contract_Tier'],
            "computed_otd": float(r['Computed_OTD']),
            "reported_otd": float(r['Reported_OTD']),
            "otd_floor_required": (float(r['OTD_Floor_Required'])
                                   if pd.notna(r['OTD_Floor_Required']) else None),
            "otd_sla_status": r['OTD_SLA_Status'],
            "otd_gap_pp": gap,
            "units_ordered": int(r['Units_Ordered']),
            "units_on_time": int(r['Units_On_Time']),
            "po_count": int(r['PO_Count']),
        },
    })

with open('./json_folder/otd_by_supplier_rag.jsonl', 'w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

print(f"Saved {len(records)} records to otd_by_supplier_rag.jsonl")

Saved 116 records to otd_by_supplier_rag.jsonl


### **3.2. Defect Rate**

In [44]:
# ============================================================
#  Defect Rate Analysis  —  Section 3.2
# ============================================================
# Tier max defect: Tier-1 <= 0.99%, Tier-2 <= 2.50%, Tier-3 <= 4.00%
# Any single shipment > 8.0% => immediate hold + RCA (flagged from per-PO max).
import json

dr = df.groupby('Supplier_ID').agg(
    Supplier_Name=('Supplier_Name', 'first'),
    Region=('Region', 'first'),
    Country=('Country', 'first'),
    Category=('Product_Category', 'first'),
    Contract_Tier=('Contract_Tier', 'first'),
    Reported_Defect=('Defect_Rate_Pct', 'mean'),
    Units_Ordered=('Units_Ordered', 'sum'),
    Units_Rejected=('Units_Rejected', 'sum'),
    Max_PO_Defect=('Defect_Rate_Pct', 'max'),
    PO_Count=('PO_ID', 'count'),
).reset_index()

dr['Computed_Defect'] = (dr['Units_Rejected'] / dr['Units_Ordered'] * 100).round(2)
dr['Reported_Defect'] = dr['Reported_Defect'].round(2)

DEFECT_MAX = {'Tier-1': 0.99, 'Tier-2': 2.50, 'Tier-3': 4.00}
dr['Defect_Max_Allowed'] = dr['Contract_Tier'].map(DEFECT_MAX)

def defect_status(r):
    cap = DEFECT_MAX.get(r['Contract_Tier'])
    if cap is None:
        return 'Unknown'
    return 'Meets SLA' if r['Computed_Defect'] <= cap else 'Exceeds SLA'

dr['Defect_SLA_Status'] = dr.apply(defect_status, axis=1)
dr['Shipment_Hold_Flag'] = dr['Max_PO_Defect'] > 8.0   # any PO >8% => hold + RCA

dr['_n'] = dr['Supplier_ID'].str.replace('SUP-', '', regex=False).astype(int)
dr = dr.sort_values('_n').drop(columns='_n').reset_index(drop=True)

print(dr['Defect_SLA_Status'].value_counts(), '\n')

Defect_SLA_Status
Meets SLA      105
Exceeds SLA     11
Name: count, dtype: int64 



In [45]:
# --- Defect RAG JSONL ---
records = []
for _, r in dr.iterrows():
    cap = r['Defect_Max_Allowed']
    margin = round(cap - r['Computed_Defect'], 2) if pd.notna(cap) else None
    hold = " A shipment exceeded the 8% threshold, triggering an immediate hold and RCA." \
           if r['Shipment_Hold_Flag'] else ""
    text = (
        f"{r['Supplier_Name']} ({r['Supplier_ID']}) has a defect rate of "
        f"{r['Computed_Defect']:.2f}% ({int(r['Units_Rejected']):,} of "
        f"{int(r['Units_Ordered']):,} units rejected at incoming inspection) across "
        f"{r['PO_Count']} purchase orders. As a {r['Contract_Tier']} supplier in "
        f"{r['Country']} ({r['Region']}) supplying {r['Category']}, the Section 3.2 maximum "
        f"is {cap:.2f}%, so the supplier {r['Defect_SLA_Status']}.{hold}"
    )
    records.append({
        "id": r['Supplier_ID'],
        "text": text,
        "metadata": {
            "supplier_id": r['Supplier_ID'], "supplier_name": r['Supplier_Name'],
            "region": r['Region'], "country": r['Country'], "category": r['Category'],
            "contract_tier": r['Contract_Tier'],
            "computed_defect": float(r['Computed_Defect']),
            "reported_defect": float(r['Reported_Defect']),
            "defect_max_allowed": (float(cap) if pd.notna(cap) else None),
            "defect_sla_status": r['Defect_SLA_Status'],
            "defect_margin_pp": margin,
            "max_po_defect": float(r['Max_PO_Defect']),
            "shipment_hold_flag": bool(r['Shipment_Hold_Flag']),
            "units_ordered": int(r['Units_Ordered']),
            "units_rejected": int(r['Units_Rejected']),
            "po_count": int(r['PO_Count']),
        },
    })

with open('./json_folder/defect_by_supplier_rag.jsonl', 'w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print(f"Saved {len(records)} records to defect_by_supplier_rag.jsonl")

Saved 116 records to defect_by_supplier_rag.jsonl


### **3.3. Lead Time Compliance**

In [46]:
# ============================================================
#  Lead Time Compliance  —  Section 3.3
# ============================================================
# Lead times > 50 calendar days => Extended Lead Time Review Protocol (ELTRP),
# regardless of tier. Flagged on the supplier's maximum observed lead time.
import json

lt = df.groupby('Supplier_ID').agg(
    Supplier_Name=('Supplier_Name', 'first'),
    Region=('Region', 'first'),
    Country=('Country', 'first'),
    Category=('Product_Category', 'first'),
    Contract_Tier=('Contract_Tier', 'first'),
    Avg_Lead_Time=('Lead_Time_Days', 'mean'),
    Max_Lead_Time=('Lead_Time_Days', 'max'),
    Min_Lead_Time=('Lead_Time_Days', 'min'),
    PO_Count=('PO_ID', 'count'),
).reset_index()

lt['Avg_Lead_Time'] = lt['Avg_Lead_Time'].round(1)
lt['ELTRP_Review_Required'] = lt['Max_Lead_Time'] > 50
lt['ELTRP_Status'] = lt['ELTRP_Review_Required'].map(
    {True: 'ELTRP Review', False: 'Within Tolerance'})

lt['_n'] = lt['Supplier_ID'].str.replace('SUP-', '', regex=False).astype(int)
lt = lt.sort_values('_n').drop(columns='_n').reset_index(drop=True)

print(lt['ELTRP_Status'].value_counts(), '\n')

ELTRP_Status
Within Tolerance    100
ELTRP Review         16
Name: count, dtype: int64 



In [47]:
# --- Lead Time RAG JSONL ---
records = []
for _, r in lt.iterrows():
    eltrp = (" The maximum lead time exceeds 50 calendar days, triggering ELTRP review."
             if r['ELTRP_Review_Required'] else "")
    text = (
        f"{r['Supplier_Name']} ({r['Supplier_ID']}) has an average lead time of "
        f"{r['Avg_Lead_Time']:.1f} calendar days (range {int(r['Min_Lead_Time'])}–"
        f"{int(r['Max_Lead_Time'])} days) across {r['PO_Count']} purchase orders. "
        f"As a {r['Contract_Tier']} supplier in {r['Country']} ({r['Region']}) supplying "
        f"{r['Category']}, the Section 3.3 ELTRP threshold is 50 calendar days, so the "
        f"supplier is {r['ELTRP_Status']}.{eltrp}"
    )
    records.append({
        "id": r['Supplier_ID'],
        "text": text,
        "metadata": {
            "supplier_id": r['Supplier_ID'], "supplier_name": r['Supplier_Name'],
            "region": r['Region'], "country": r['Country'], "category": r['Category'],
            "contract_tier": r['Contract_Tier'],
            "avg_lead_time_days": float(r['Avg_Lead_Time']),
            "max_lead_time_days": int(r['Max_Lead_Time']),
            "min_lead_time_days": int(r['Min_Lead_Time']),
            "eltrp_review_required": bool(r['ELTRP_Review_Required']),
            "eltrp_status": r['ELTRP_Status'],
            "po_count": int(r['PO_Count']),
        },
    })

with open('./json_folder/leadtime_by_supplier_rag.jsonl', 'w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print(f"Saved {len(records)} records to leadtime_by_supplier_rag.jsonl")

Saved 116 records to leadtime_by_supplier_rag.jsonl


### **3.4. Compliance Score Floor**

In [48]:
# ============================================================
#  Compliance Score Floor  —  Section 3.4 (with Section 7.2)
# ============================================================
# Section 3.4: Compliance Score < 60 at any audit => Supplier Watch List (SWL),
#              regardless of tier (limits new PO issuance to 20% of prior quarter).
# Section 7.2: Score < 70 at a scheduled audit => mandatory follow-up audit.
# Flagged on the supplier's MINIMUM observed score (worst audit).
import json

cs = df.groupby('Supplier_ID').agg(
    Supplier_Name=('Supplier_Name', 'first'),
    Region=('Region', 'first'),
    Country=('Country', 'first'),
    Category=('Product_Category', 'first'),
    Contract_Tier=('Contract_Tier', 'first'),
    Avg_Compliance=('Compliance_Score', 'mean'),
    Min_Compliance=('Compliance_Score', 'min'),
    PO_Count=('PO_ID', 'count'),
).reset_index()

cs['Avg_Compliance'] = cs['Avg_Compliance'].round(2)
cs['SWL_Flag'] = cs['Min_Compliance'] < 60            # below 60 at any audit
cs['Followup_Audit_Flag'] = cs['Min_Compliance'] < 70 # below 70 => follow-up audit

def compliance_status(r):
    if r['SWL_Flag']:
        return 'SWL (below 60)'
    if r['Followup_Audit_Flag']:
        return 'Follow-up Audit (below 70)'
    return 'OK'

cs['Compliance_Status'] = cs.apply(compliance_status, axis=1)

cs['_n'] = cs['Supplier_ID'].str.replace('SUP-', '', regex=False).astype(int)
cs = cs.sort_values('_n').drop(columns='_n').reset_index(drop=True)

print(cs['Compliance_Status'].value_counts(), '\n')

Compliance_Status
OK                            47
Follow-up Audit (below 70)    45
SWL (below 60)                24
Name: count, dtype: int64 



In [49]:
# --- Compliance RAG JSONL ---
records = []
for _, r in cs.iterrows():
    if r['SWL_Flag']:
        note = (" Because a compliance score below 60 was recorded, the supplier is placed "
                "on the Supplier Watch List (SWL), limiting new PO issuance to 20% of prior "
                "quarter volume.")
    elif r['Followup_Audit_Flag']:
        note = (" A compliance score below 70 was recorded at audit, triggering a mandatory "
                "follow-up audit within 60 days at the supplier's cost.")
    else:
        note = " All recorded compliance scores are at or above the required floors."
    text = (
        f"{r['Supplier_Name']} ({r['Supplier_ID']}) has an average compliance score of "
        f"{r['Avg_Compliance']:.1f} (lowest recorded {int(r['Min_Compliance'])}) across "
        f"{r['PO_Count']} purchase orders. As a {r['Contract_Tier']} supplier in "
        f"{r['Country']} ({r['Region']}) supplying {r['Category']}, the Section 3.4 status "
        f"is {r['Compliance_Status']}.{note}"
    )
    records.append({
        "id": r['Supplier_ID'],
        "text": text,
        "metadata": {
            "supplier_id": r['Supplier_ID'], "supplier_name": r['Supplier_Name'],
            "region": r['Region'], "country": r['Country'], "category": r['Category'],
            "contract_tier": r['Contract_Tier'],
            "avg_compliance": float(r['Avg_Compliance']),
            "min_compliance": int(r['Min_Compliance']),
            "swl_flag": bool(r['SWL_Flag']),
            "followup_audit_flag": bool(r['Followup_Audit_Flag']),
            "compliance_status": r['Compliance_Status'],
            "po_count": int(r['PO_Count']),
        },
    })

with open('./json_folder/compliance_by_supplier_rag.jsonl', 'w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print(f"Saved {len(records)} records to compliance_by_supplier_rag.jsonl")

Saved 116 records to compliance_by_supplier_rag.jsonl


In [50]:
# # ============================================================
# #  Section 3 SLA checks (OTD, Defect, Lead Time, Compliance)
# # ============================================================
# import json

# g = df.groupby('Supplier_ID').agg(
#     Supplier_Name=('Supplier_Name', 'first'), Region=('Region', 'first'),
#     Country=('Country', 'first'), Category=('Product_Category', 'first'),
#     Contract_Tier=('Contract_Tier', 'first'), PO_Count=('PO_ID', 'count'),
#     UOrd=('Units_Ordered', 'sum'), UOnTime=('Units_Delivered_On_Time', 'sum'),
#     URej=('Units_Rejected', 'sum'),
#     MaxDefect=('Defect_Rate_Pct', 'max'),
#     AvgLead=('Lead_Time_Days', 'mean'), MaxLead=('Lead_Time_Days', 'max'),
#     AvgComp=('Compliance_Score', 'mean'), MinComp=('Compliance_Score', 'min'),
# ).reset_index()

# # --- derived metrics (recomputed from raw units where applicable) ---
# g['OTD']     = (g.UOnTime / g.UOrd * 100).round(2)
# g['Defect']  = (g.URej   / g.UOrd * 100).round(2)
# g['AvgLead'] = g.AvgLead.round(1)
# g['AvgComp'] = g.AvgComp.round(2)

# # --- Section 3 thresholds & status ---
# OTD_FLOOR = {'Tier-1': 93.0, 'Tier-2': 84.0, 'Tier-3': 75.0}   # 3.1
# DEF_MAX   = {'Tier-1': 0.99, 'Tier-2': 2.50, 'Tier-3': 4.00}   # 3.2

# g['OTD_Status']        = g.apply(lambda r: 'Meets SLA' if r.OTD >= OTD_FLOOR.get(r.Contract_Tier, 1e9) else 'Below SLA', axis=1)
# g['Defect_Status']     = g.apply(lambda r: 'Meets SLA' if r.Defect <= DEF_MAX.get(r.Contract_Tier, -1) else 'Exceeds SLA', axis=1)
# g['Hold_Flag']         = g.MaxDefect > 8.0                       # 3.2: any PO >8% => hold+RCA
# g['ELTRP_Status']      = (g.MaxLead > 50).map({True: 'ELTRP Review', False: 'Within Tolerance'})  # 3.3
# g['Compliance_Status'] = g.apply(lambda r: 'SWL (below 60)' if r.MinComp < 60
#                                  else ('Follow-up Audit (below 70)' if r.MinComp < 70 else 'OK'), axis=1)  # 3.4

# # --- one passage per metric ---
# def make_text(r, m):
#     s = f"{r.Supplier_Name} ({r.Supplier_ID}), {r.Contract_Tier}, {r.Country} ({r.Region}), {r.Category}. "
#     if m == 'otd':    return s + f"OTD {r.OTD:.1f}% over {r.PO_Count} POs; Section 3.1 floor {OTD_FLOOR.get(r.Contract_Tier)}% -> {r.OTD_Status}."
#     if m == 'defect': return s + f"Defect rate {r.Defect:.2f}%; Section 3.2 max {DEF_MAX.get(r.Contract_Tier)}% -> {r.Defect_Status}." + (" Shipment >8% triggered hold+RCA." if r.Hold_Flag else "")
#     if m == 'lead':   return s + f"Avg lead time {r.AvgLead:.1f} days (max {int(r.MaxLead)}); Section 3.3 ELTRP at 50 days -> {r.ELTRP_Status}."
#     if m == 'comp':   return s + f"Avg compliance {r.AvgComp:.1f} (min {int(r.MinComp)}); Section 3.4 -> {r.Compliance_Status}."

# # metric -> the extra metadata columns it should carry
# METRICS = {
#     'otd':    ['OTD', 'OTD_Status'],
#     'defect': ['Defect', 'Defect_Status', 'Hold_Flag'],
#     'lead':   ['AvgLead', 'MaxLead', 'ELTRP_Status'],
#     'comp':   ['AvgComp', 'MinComp', 'Compliance_Status'],
# }
# common = ['Supplier_ID', 'Supplier_Name', 'Region', 'Country', 'Category', 'Contract_Tier', 'PO_Count']

# for m, cols in METRICS.items():
#     with open(f'{m}_rag.jsonl', 'w', encoding='utf-8') as f:
#         for _, r in g.iterrows():
#             meta = {c.lower(): (r[c].item() if hasattr(r[c], 'item') else r[c]) for c in common + cols}
#             rec = {"id": r.Supplier_ID, "text": make_text(r, m), "metadata": meta}
#             f.write(json.dumps(rec, ensure_ascii=False, default=str) + '\n')
#     print(f"Saved {m}_rag.jsonl")

## **4. Penalty and Incentive Structure**

### **4.1. OTD Penalty Tiers**

In [51]:
# ============================================================
#  Section 4.1 — OTD Penalty Tiers (assessed per supplier-quarter)
# ============================================================
import json

# Parse "Q1-2023" -> quarter/year for ordering
qp = df['PO_Quarter'].str.extract(r'Q(?P<q>\d)-(?P<y>\d{4})')
df['Q'] = qp['q'].astype(int)
df['Y'] = qp['y'].astype(int)

# Tier-specific penalty bands (note: several bands apply to Tier-1 ONLY)
def otd_penalty_rate(otd, tier):
    if otd < 70.0: return 8.0                                   # all tiers + CAP
    if otd < 75.0: return 5.0 if tier == 'Tier-1' else 0.0
    if otd < 80.0: return 3.5 if tier in ('Tier-1', 'Tier-2') else 0.0
    if otd < 84.0: return 2.0 if tier == 'Tier-1' else 0.0
    if otd < 87.0: return 1.0 if tier == 'Tier-1' else 0.0
    return 0.0                                                  # >=87% no penalty

sq = df.groupby(['Supplier_ID', 'PO_Quarter', 'Y', 'Q']).agg(
    Supplier_Name=('Supplier_Name', 'first'), Region=('Region', 'first'),
    Tier=('Contract_Tier', 'first'),
    OTD=('OTD_Rate_Pct', 'mean'),
    Q_Invoice=('PO_Value_USD', 'sum'),
).reset_index()

sq['OTD'] = sq.OTD.round(2)
sq['Penalty_Rate'] = sq.apply(lambda r: otd_penalty_rate(r.OTD, r.Tier), axis=1)
sq['Penalty_USD'] = (sq.Penalty_Rate / 100 * sq.Q_Invoice).round(2)
sq['CAP_Required'] = sq.OTD < 70.0          # mandatory CAP within 10 days

sq = sq.sort_values(['Supplier_ID', 'Y', 'Q']).reset_index(drop=True)

print(f"{(sq.Penalty_Rate>0).sum()} penalised quarters | "
      f"${sq.Penalty_USD.sum():,.0f} total | {int(sq.CAP_Required.sum())} CAP triggers")

# --- RAG JSONL (only penalised quarters are worth indexing) ---
with open('./json_folder/otd_penalties_rag.jsonl', 'w', encoding='utf-8') as f:
    for _, r in sq[sq.Penalty_Rate > 0].iterrows():
        cap = " A CAP is mandatory within 10 days." if r.CAP_Required else ""
        text = (f"{r.Supplier_Name} ({r.Supplier_ID}), {r.Tier}, recorded OTD of {r.OTD:.1f}% "
                f"in {r.PO_Quarter}. Under Section 4.1 this incurs a {r.Penalty_Rate:.1f}% "
                f"penalty on the quarterly invoice (${r.Q_Invoice:,.0f}), i.e. "
                f"${r.Penalty_USD:,.0f}, applied as a deduction next quarter.{cap}")
        f.write(json.dumps({
            "id": f"{r.Supplier_ID}_{r.PO_Quarter}_otdpen",
            "text": text,
            "metadata": {
                "supplier_id": r.Supplier_ID, "supplier_name": r.Supplier_Name,
                "region": r.Region, "tier": r.Tier, "quarter": r.PO_Quarter,
                "otd_pct": float(r.OTD), "penalty_rate_pct": float(r.Penalty_Rate),
                "quarterly_invoice_usd": float(r.Q_Invoice),
                "penalty_usd": float(r.Penalty_USD), "cap_required": bool(r.CAP_Required),
            },
        }, ensure_ascii=False) + '\n')
print("Saved otd_penalties_rag.jsonl")

57 penalised quarters | $2,028,912 total | 45 CAP triggers
Saved otd_penalties_rag.jsonl


### **4.2. Volume Rebate Program (Tier-1 only)**

In [52]:
# ============================================================
#  Section 4.2 — Volume Rebate Program (Tier-1, full calendar year)
# ============================================================
# Tier-1 AND year OTD >= 93% AND defect < 0.5% AND sustainability >= 85
#   => 2.5% of total annual invoice value.
import json

yr = df.groupby(['Supplier_ID', 'Y']).agg(
    Supplier_Name=('Supplier_Name', 'first'), Region=('Region', 'first'),
    Tier=('Contract_Tier', 'first'),
    OTD=('OTD_Rate_Pct', 'mean'),
    Defect=('Defect_Rate_Pct', 'mean'),
    Sustain=('Sustainability_Score', 'mean'),
    Annual_Invoice=('PO_Value_USD', 'sum'),
).reset_index()

for c in ['OTD', 'Defect', 'Sustain']:
    yr[c] = yr[c].round(2)

yr['Rebate_Eligible'] = ((yr.Tier == 'Tier-1') & (yr.OTD >= 93.0) &
                         (yr.Defect < 0.5) & (yr.Sustain >= 85.0))
yr['Rebate_USD'] = (yr.Rebate_Eligible * 0.025 * yr.Annual_Invoice).round(2)

yr = yr.sort_values(['Supplier_ID', 'Y']).reset_index(drop=True)

print(f"{int(yr.Rebate_Eligible.sum())} eligible supplier-years | "
      f"${yr.Rebate_USD.sum():,.0f} total rebate")

# --- RAG JSONL (index all Tier-1 supplier-years, eligible or not, with the reason) ---
with open('./json_folder/volume_rebate_rag.jsonl', 'w', encoding='utf-8') as f:
    for _, r in yr[yr.Tier == 'Tier-1'].iterrows():
        if r.Rebate_Eligible:
            verdict = (f"qualifies for a 2.5% volume rebate of ${r.Rebate_USD:,.0f}")
        else:
            fails = []
            if r.OTD < 93.0:     fails.append(f"OTD {r.OTD:.1f}% (<93%)")
            if r.Defect >= 0.5:  fails.append(f"defect {r.Defect:.2f}% (>=0.5%)")
            if r.Sustain < 85.0: fails.append(f"sustainability {r.Sustain:.0f} (<85)")
            verdict = "is not eligible for the volume rebate, failing on " + ", ".join(fails)
        text = (f"{r.Supplier_Name} ({r.Supplier_ID}), a Tier-1 supplier, in {int(r.Y)} "
                f"had OTD {r.OTD:.1f}%, defect {r.Defect:.2f}%, sustainability {r.Sustain:.0f}, "
                f"and annual invoice ${r.Annual_Invoice:,.0f}. Under Section 4.2 it {verdict}.")
        f.write(json.dumps({
            "id": f"{r.Supplier_ID}_{int(r.Y)}_rebate",
            "text": text,
            "metadata": {
                "supplier_id": r.Supplier_ID, "supplier_name": r.Supplier_Name,
                "region": r.Region, "tier": r.Tier, "year": int(r.Y),
                "otd_pct": float(r.OTD), "defect_pct": float(r.Defect),
                "sustainability": float(r.Sustain),
                "annual_invoice_usd": float(r.Annual_Invoice),
                "rebate_eligible": bool(r.Rebate_Eligible),
                "rebate_usd": float(r.Rebate_USD),
            },
        }, ensure_ascii=False) + '\n')
print("Saved volume_rebate_rag.jsonl")

0 eligible supplier-years | $0 total rebate
Saved volume_rebate_rag.jsonl


### **4.3. Defect Penalty Escalation**

In [53]:
# ============================================================
#  Section 4.3 — Defect Penalty Escalation (per supplier-quarter)
# ============================================================
# Defect > tier max for TWO CONSECUTIVE quarters => +4.0% surcharge on the affected quarter.
# Any single shipment > 5.0% => Quality Containment Fee of $15,000 per occurrence.
import json

DEF_MAX = {'Tier-1': 0.99, 'Tier-2': 2.50, 'Tier-3': 4.00}

sqd = df.groupby(['Supplier_ID', 'PO_Quarter', 'Y', 'Q']).agg(
    Supplier_Name=('Supplier_Name', 'first'), Region=('Region', 'first'),
    Tier=('Contract_Tier', 'first'),
    Defect=('Defect_Rate_Pct', 'mean'),
    Q_Invoice=('PO_Value_USD', 'sum'),
    QCF_Count=('Defect_Rate_Pct', lambda s: int((s > 5.0).sum())),   # shipments >5%
).reset_index().sort_values(['Supplier_ID', 'Y', 'Q'])

sqd['Defect'] = sqd.Defect.round(2)
sqd['Over_Max'] = sqd.apply(lambda r: r.Defect > DEF_MAX.get(r.Tier, 1e9), axis=1)
# consecutive = this quarter over AND the supplier's previous chronological quarter over
sqd['Prev_Over'] = sqd.groupby('Supplier_ID')['Over_Max'].shift(1).fillna(False)
sqd['Surcharge_Applies'] = sqd.Over_Max & sqd.Prev_Over
sqd['Surcharge_USD'] = (sqd.Surcharge_Applies * 0.04 * sqd.Q_Invoice).round(2)
sqd['QCF_USD'] = sqd.QCF_Count * 15000

sqd = sqd.reset_index(drop=True)

print(f"{int(sqd.Surcharge_Applies.sum())} surcharge quarters (${sqd.Surcharge_USD.sum():,.0f}) | "
      f"{int(sqd.QCF_Count.sum())} QCF occurrences (${sqd.QCF_USD.sum():,.0f})")

# --- RAG JSONL (index quarters with a surcharge or a QCF) ---
flagged = sqd[(sqd.Surcharge_Applies) | (sqd.QCF_Count > 0)]
with open('./json_folder/defect_escalation_rag.jsonl', 'w', encoding='utf-8') as f:
    for _, r in flagged.iterrows():
        parts = []
        if r.Surcharge_Applies:
            parts.append(f"a 4.0% surcharge of ${r.Surcharge_USD:,.0f} for exceeding the "
                         f"{DEF_MAX[r.Tier]:.2f}% tier maximum two consecutive quarters")
        if r.QCF_Count > 0:
            parts.append(f"{int(r.QCF_Count)} Quality Containment Fee(s) of $15,000 each "
                         f"(${r.QCF_USD:,.0f}) for shipment(s) above 5%")
        text = (f"{r.Supplier_Name} ({r.Supplier_ID}), {r.Tier}, in {r.PO_Quarter} had a "
                f"defect rate of {r.Defect:.2f}% and incurs " + " and ".join(parts) +
                " under Section 4.3.")
        f.write(json.dumps({
            "id": f"{r.Supplier_ID}_{r.PO_Quarter}_defesc",
            "text": text,
            "metadata": {
                "supplier_id": r.Supplier_ID, "supplier_name": r.Supplier_Name,
                "region": r.Region, "tier": r.Tier, "quarter": r.PO_Quarter,
                "defect_pct": float(r.Defect),
                "surcharge_applies": bool(r.Surcharge_Applies),
                "surcharge_usd": float(r.Surcharge_USD),
                "qcf_count": int(r.QCF_Count), "qcf_usd": float(r.QCF_USD),
                "quarterly_invoice_usd": float(r.Q_Invoice),
            },
        }, ensure_ascii=False) + '\n')
print("Saved defect_escalation_rag.jsonl")

45 surcharge quarters ($357,031) | 0 QCF occurrences ($0)
Saved defect_escalation_rag.jsonl


## **5. Risk Assessment and Escalation Protocol**

### **5.1. Risk Level Definitions**

In [54]:
# ============================================================
#  Section 5.1 — Risk Level (recomputed from disruption + compliance factors)
# ============================================================
# Audit-age factor intentionally EXCLUDED (dataset audit dates predate the
# policy reference date, which would force every supplier to "overdue").
import json

qp = df['PO_Quarter'].str.extract(r'Q(?P<q>\d)-(?P<y>\d{4})')
df['Q'] = qp['q'].astype(int)
df['Y'] = qp['y'].astype(int)

def disruption_factors(text):
    """Map the Active_Disruptions free-text into Section 5.1 factor categories."""
    d = str(text).lower(); f = set()
    if 'geopolitical' in d:                                          f.add('geopolitical')
    if any(k in d for k in ['monsoon', 'typhoon', 'flood', 'seasonal']): f.add('seasonal')
    if 'regulatory audit pending' in d or 'compliance review' in d or 'audit overdue' in d:
                                                                     f.add('pending_audit')
    if 'tariff' in d:                                                f.add('tariff')
    if 'labour' in d or 'strike' in d:                              f.add('labour')
    if 'export' in d:                                                f.add('export_control')
    if 'currency' in d:                                             f.add('currency')
    if 'port closure' in d:                                          f.add('port_closure')
    return f

def risk_level(disruption_text, comp):
    f = disruption_factors(disruption_text)
    # HIGH if any single severe factor, or compliance below 75
    high_single = ({'labour', 'export_control', 'currency'} & f) or (comp < 75)
    # MEDIUM factors
    med = {k for k in ('geopolitical', 'seasonal', 'pending_audit', 'tariff') if k in f}
    if 75 <= comp <= 85:
        med.add('comp_75_85')
    # HIGH also if two or more medium factors co-occur
    if high_single or len(med) >= 2:
        return 'High'
    if len(med) >= 1:
        return 'Medium'
    return 'Low'

# Per supplier-quarter risk (needed for escalation in 5.2)
sq_risk = df.groupby(['Supplier_ID', 'PO_Quarter', 'Y', 'Q']).agg(
    Supplier_Name=('Supplier_Name', 'first'), Region=('Region', 'first'),
    Country=('Country', 'first'), Tier=('Contract_Tier', 'first'),
    Disruption=('Active_Disruptions', 'first'),
    Comp=('Compliance_Score', 'mean'),
    Reported_Risk=('Risk_Level', 'first'),
).reset_index().sort_values(['Supplier_ID', 'Y', 'Q'])

sq_risk['Comp'] = sq_risk.Comp.round(2)
sq_risk['Computed_Risk'] = sq_risk.apply(lambda r: risk_level(r.Disruption, r.Comp), axis=1)
sq_risk['Risk_Factors'] = sq_risk.apply(
    lambda r: sorted(disruption_factors(r.Disruption) |
                     ({'comp_below_75'} if r.Comp < 75 else set())), axis=1)

# Latest-quarter snapshot = current risk per supplier
risk_now = sq_risk.groupby('Supplier_ID').tail(1).reset_index(drop=True)

print("Current risk:", risk_now.Computed_Risk.value_counts().to_dict())

# --- RAG JSONL: current risk per supplier ---
with open('./json_folder/risk_levels_rag.jsonl', 'w', encoding='utf-8') as f:
    for _, r in risk_now.iterrows():
        disr = r.Disruption if str(r.Disruption) != 'None' else 'no active disruption'
        factors_str = ', '.join(r.Risk_Factors) if r.Risk_Factors else 'none'
        text = (f"{r.Supplier_Name} ({r.Supplier_ID}), {r.Tier}, {r.Country} ({r.Region}), "
                f"has a Section 5.1 risk level of {r.Computed_Risk} as of {r.PO_Quarter}. "
                f"Compliance score {r.Comp:.0f}; active disruption: {disr}. "
                f"Contributing risk factors: {factors_str}. "
                f"(CSV-reported risk: {r.Reported_Risk}.)")
        f.write(json.dumps({
            "id": f"{r.Supplier_ID}_risk",
            "text": text,
            "metadata": {
                "supplier_id": r.Supplier_ID, "supplier_name": r.Supplier_Name,
                "region": r.Region, "country": r.Country, "tier": r.Tier,
                "quarter": r.PO_Quarter, "computed_risk": r.Computed_Risk,
                "reported_risk": r.Reported_Risk, "compliance_score": float(r.Comp),
                "active_disruption": str(r.Disruption), "risk_factors": r.Risk_Factors,
            },
        }, ensure_ascii=False) + '\n')
print("Saved risk_levels_rag.jsonl")

Current risk: {'High': 48, 'Low': 36, 'Medium': 32}
Saved risk_levels_rag.jsonl


### **5.2. Escalation Triggers**

In [55]:
# ============================================================
#  Section 5.2 — Escalation Triggers (High risk 2 consecutive quarters => CPO review)
# ============================================================
# CPO review outcome is one of: (a) Remediation Plan, (b) Parallel Sourcing,
# (c) Supplier Exit (90-day wind-down). The dataset can't determine which —
# we flag that a CPO strategic review is REQUIRED.
import json

sq_risk['High'] = sq_risk.Computed_Risk == 'High'
sq_risk['Prev_High'] = sq_risk.groupby('Supplier_ID')['High'].shift(1).fillna(False)
sq_risk['Escalate_CPO'] = sq_risk.High & sq_risk.Prev_High

escal = sq_risk[sq_risk.Escalate_CPO].copy()

print(f"{len(escal)} CPO escalation events across {escal.Supplier_ID.nunique()} suppliers")

# --- RAG JSONL: one record per escalation event (supplier-quarter) ---
with open('./json_folder/escalations_rag.jsonl', 'w', encoding='utf-8') as f:
    for _, r in escal.iterrows():
        text = (f"{r.Supplier_Name} ({r.Supplier_ID}), {r.Tier}, {r.Country} ({r.Region}), "
                f"was High risk for two consecutive quarters (through {r.PO_Quarter}), "
                f"triggering mandatory escalation to the Chief Procurement Officer under "
                f"Section 5.2 for strategic review (remediation plan, parallel sourcing, "
                f"or supplier exit). Active disruption: {r.Disruption}; compliance {r.Comp:.0f}.")
        f.write(json.dumps({
            "id": f"{r.Supplier_ID}_{r.PO_Quarter}_escal",
            "text": text,
            "metadata": {
                "supplier_id": r.Supplier_ID, "supplier_name": r.Supplier_Name,
                "region": r.Region, "country": r.Country, "tier": r.Tier,
                "quarter": r.PO_Quarter, "escalation": "CPO strategic review",
                "computed_risk": r.Computed_Risk, "compliance_score": float(r.Comp),
                "active_disruption": str(r.Disruption),
            },
        }, ensure_ascii=False) + '\n')
print("Saved escalations_rag.jsonl")

209 CPO escalation events across 60 suppliers
Saved escalations_rag.jsonl


### **5.3. Concentration Risk Rule**

In [56]:
# ============================================================
#  Section 5.3 — Concentration Risk (portfolio-level, not per supplier)
# ============================================================
# No single region > 45% of total annual spend; no single country > 25%.
# Breach => Diversification Plan required within 60 days.
import json

total_spend = df['PO_Value_USD'].sum()

region_share = (df.groupby('Region')['PO_Value_USD'].sum() / total_spend * 100).round(2)
country_share = (df.groupby('Country')['PO_Value_USD'].sum() / total_spend * 100).round(2)

region_df = region_share.reset_index().rename(columns={'PO_Value_USD': 'Spend_Share_Pct'})
region_df['Limit_Pct'] = 45.0
region_df['Breach'] = region_df.Spend_Share_Pct > 45.0
region_df['Scope'] = 'region'

country_df = country_share.reset_index().rename(columns={'PO_Value_USD': 'Spend_Share_Pct'})
country_df['Limit_Pct'] = 25.0
country_df['Breach'] = country_df.Spend_Share_Pct > 25.0
country_df['Scope'] = 'country'

conc = pd.concat([
    region_df.rename(columns={'Region': 'Entity'}),
    country_df.rename(columns={'Country': 'Entity'}),
], ignore_index=True).sort_values(['Scope', 'Spend_Share_Pct'], ascending=[True, False])


print("Region shares:", region_share.sort_values(ascending=False).to_dict())
print("Any breach:", bool(conc.Breach.any()),
      "| Diversification Plan required:", bool(conc.Breach.any()))

# --- RAG JSONL: one record per region/country share + a portfolio summary ---
with open('./json_folder/concentration_risk_rag.jsonl', 'w', encoding='utf-8') as f:
    for _, r in conc.iterrows():
        verdict = ("EXCEEDS the limit, requiring a Diversification Plan within 60 days"
                   if r.Breach else "is within the limit")
        text = (f"In the BQBYTE supply chain, {r.Scope} '{r.Entity}' accounts for "
                f"{r.Spend_Share_Pct:.1f}% of total annual procurement spend. Under Section "
                f"5.3 the {r.Scope} concentration limit is {r.Limit_Pct:.0f}%, so this {verdict}.")
        f.write(json.dumps({
            "id": f"conc_{r.Scope}_{r.Entity}".replace(' ', '_'),
            "text": text,
            "metadata": {
                "scope": r.Scope, "entity": r.Entity,
                "spend_share_pct": float(r.Spend_Share_Pct),
                "limit_pct": float(r.Limit_Pct), "breach": bool(r.Breach),
            },
        }, ensure_ascii=False) + '\n')

    # portfolio-level summary record
    top_r = region_share.idxmax(); top_c = country_share.idxmax()
    summary = (f"Across the BQBYTE portfolio, the most concentrated region is {top_r} at "
               f"{region_share.max():.1f}% (limit 45%) and the most concentrated country is "
               f"{top_c} at {country_share.max():.1f}% (limit 25%). No concentration limit is "
               f"currently breached, so no Diversification Plan is required under Section 5.3.")
    f.write(json.dumps({
        "id": "conc_portfolio_summary", "text": summary,
        "metadata": {"scope": "portfolio", "top_region": top_r,
                     "top_region_pct": float(region_share.max()), "top_country": top_c,
                     "top_country_pct": float(country_share.max()),
                     "any_breach": bool(conc.Breach.any())},
    }, ensure_ascii=False) + '\n')
print("Saved concentration_risk_rag.jsonl")

Region shares: {'APAC': 36.97, 'EMEA': 26.44, 'LATAM': 23.58, 'Unknown': 13.01}
Any breach: False | Diversification Plan required: False
Saved concentration_risk_rag.jsonl


## **6. Sustainability and Compliance Requirements**

### **6.1. Sustainability Score**

In [57]:
# ============================================================
#  Section 6.1 — Sustainability Score Floor
# ============================================================
# Floors by tier: Tier-1 >= 80, Tier-2 >= 60, Tier-3 >= 45.
# Below floor => Sustainability Improvement Plan (SIP) within 45 days of audit notice.
import json

SUS_FLOOR = {'Tier-1': 80, 'Tier-2': 60, 'Tier-3': 45}

sus = df.groupby('Supplier_ID').agg(
    Supplier_Name=('Supplier_Name', 'first'), Region=('Region', 'first'),
    Country=('Country', 'first'), Category=('Product_Category', 'first'),
    Tier=('Contract_Tier', 'first'),
    Sustain=('Sustainability_Score', 'mean'),
    Min_Sustain=('Sustainability_Score', 'min'),
    PO_Count=('PO_ID', 'count'),
).reset_index()

sus['Sustain'] = sus.Sustain.round(1)
sus['Floor_Required'] = sus.Tier.map(SUS_FLOOR)
sus['SIP_Required'] = sus.Sustain < sus.Floor_Required
sus['Sustain_Gap'] = (sus.Sustain - sus.Floor_Required).round(1)

sus['_n'] = sus.Supplier_ID.str.replace('SUP-', '', regex=False).astype(int)
sus = sus.sort_values('_n').drop(columns='_n').reset_index(drop=True)

print(f"{int(sus.SIP_Required.sum())} suppliers require a Sustainability Improvement Plan")

# --- RAG JSONL ---
with open('./json_folder/sustainability_rag.jsonl', 'w', encoding='utf-8') as f:
    for _, r in sus.iterrows():
        verdict = (f"falls below the floor by {abs(r.Sustain_Gap):.1f} points and must submit "
                   f"a Sustainability Improvement Plan within 45 days"
                   if r.SIP_Required else
                   f"meets the floor (margin {r.Sustain_Gap:+.1f} points)")
        text = (f"{r.Supplier_Name} ({r.Supplier_ID}), {r.Tier}, {r.Country} ({r.Region}), "
                f"supplying {r.Category}, has an average sustainability score of "
                f"{r.Sustain:.1f} (lowest {int(r.Min_Sustain)}). Under Section 6.1 the "
                f"{r.Tier} floor is {r.Floor_Required}, so the supplier {verdict}.")
        f.write(json.dumps({
            "id": f"{r.Supplier_ID}_sustain",
            "text": text,
            "metadata": {
                "supplier_id": r.Supplier_ID, "supplier_name": r.Supplier_Name,
                "region": r.Region, "country": r.Country, "category": r.Category,
                "tier": r.Tier, "sustainability_score": float(r.Sustain),
                "min_sustainability": int(r.Min_Sustain),
                "floor_required": int(r.Floor_Required),
                "sip_required": bool(r.SIP_Required),
                "sustainability_gap": float(r.Sustain_Gap),
            },
        }, ensure_ascii=False) + '\n')
print("Saved sustainability_rag.jsonl")

7 suppliers require a Sustainability Improvement Plan
Saved sustainability_rag.jsonl


### 6.2. Mandatory Certification Requirements**

In [58]:
# ============================================================
#  Section 6.2 — Mandatory Certification Requirements
# ============================================================
# Universal: ISO 9001 (all suppliers, all tiers).
# Category-conditional (rules verifiable from the data):
#   EC -> RoHS                         RM(EMEA) -> REACH; Tier-1 RM -> ISO14001
#   PM -> FSC; Tier-1 PM -> ISO14001   IT -> OEKO-TEX     Tier-1 SA -> AS9100
# Conditional/recommended certs that depend on END-USE (aerospace/automotive) are
# NOT enforced, because the dataset cannot establish end-use:
#   NADCAP, AS9100(non-Tier1), IATF 16949, AEC-Q100  -> recorded as advisory only.
# REACH proxied by EMEA region (no destination-market field in the data).
import json

CAT = {'Electronic Components': 'EC', 'Raw Materials': 'RM',
       'Mechanical Components': 'MC', 'Packaging Materials': 'PM',
       'Industrial Textiles': 'IT', 'Specialty Alloys': 'SA'}

def parse_certs(s):
    return set(x.strip() for x in str(s).split(';') if x.strip())

# Union of certs across a supplier's POs (guards against per-PO inconsistency)
cert_union = df.groupby('Supplier_ID')['Certifications'].apply(
    lambda col: set().union(*[parse_certs(v) for v in col]))

cert = df.groupby('Supplier_ID').agg(
    Supplier_Name=('Supplier_Name', 'first'), Region=('Region', 'first'),
    Country=('Country', 'first'), Category=('Product_Category', 'first'),
    Tier=('Contract_Tier', 'first'),
).reset_index()
cert['Held_Certs'] = cert.Supplier_ID.map(cert_union)

def required_certs(tier, cat_full, region):
    cat = CAT[cat_full]
    req = {'ISO9001'}                                   # universal baseline
    if cat == 'EC':
        req |= {'RoHS'}
    if cat == 'RM':
        if region == 'EMEA':                            # EU-market proxy
            req |= {'REACH'}
        if tier == 'Tier-1':
            req |= {'ISO14001'}
    if cat == 'PM':
        req |= {'FSC'}
        if tier == 'Tier-1':
            req |= {'ISO14001'}
    if cat == 'IT':
        req |= {'OEKO-TEX'}
    if cat == 'SA' and tier == 'Tier-1':
        req |= {'AS9100'}
    return req

# Advisory (end-use conditional) certs — reported, not enforced
def advisory_certs(cat_full):
    cat = CAT[cat_full]
    return {
        'EC': ['AEC-Q100 (if automotive)'],
        'MC': ['IATF 16949 (if automotive)'],
        'SA': ['NADCAP (if aerospace)'],
    }.get(cat, [])

cert['Required_Certs'] = cert.apply(lambda r: required_certs(r.Tier, r.Category, r.Region), axis=1)
cert['Missing_Certs'] = cert.apply(lambda r: sorted(r.Required_Certs - r.Held_Certs), axis=1)
cert['Advisory_Certs'] = cert.Category.map(advisory_certs)
cert['Cert_Compliant'] = cert.Missing_Certs.apply(len) == 0

cert['_n'] = cert.Supplier_ID.str.replace('SUP-', '', regex=False).astype(int)
cert = cert.sort_values('_n').drop(columns='_n').reset_index(drop=True)

# Sets aren't CSV/JSON-friendly -> stringify for the CSV copy
cert_csv = cert.copy()
cert_csv['Held_Certs'] = cert_csv.Held_Certs.apply(lambda s: ';'.join(sorted(s)))
cert_csv['Required_Certs'] = cert_csv.Required_Certs.apply(lambda s: ';'.join(sorted(s)))
cert_csv['Missing_Certs'] = cert_csv.Missing_Certs.apply(lambda l: ';'.join(l))
cert_csv['Advisory_Certs'] = cert_csv.Advisory_Certs.apply(lambda l: ';'.join(l))

print(f"{int(cert.Cert_Compliant.sum())} compliant | {int((~cert.Cert_Compliant).sum())} missing mandatory certs")

# --- RAG JSONL ---
with open('./json_folder/certifications_rag.jsonl', 'w', encoding='utf-8') as f:
    for _, r in cert.iterrows():
        held = ', '.join(sorted(r.Held_Certs)) or 'none'
        req = ', '.join(sorted(r.Required_Certs))
        if r.Cert_Compliant:
            verdict = "holds all mandatory certifications for its category and tier"
        else:
            verdict = f"is MISSING mandatory certification(s): {', '.join(r.Missing_Certs)}"
        adv = (f" Advisory (end-use dependent): {', '.join(r.Advisory_Certs)}."
               if r.Advisory_Certs else "")
        text = (f"{r.Supplier_Name} ({r.Supplier_ID}), {r.Tier}, {r.Country} ({r.Region}), "
                f"supplying {r.Category}, holds {held}. Under Section 6.2 the mandatory set is "
                f"{req}. The supplier {verdict}.{adv}")
        f.write(json.dumps({
            "id": f"{r.Supplier_ID}_certs",
            "text": text,
            "metadata": {
                "supplier_id": r.Supplier_ID, "supplier_name": r.Supplier_Name,
                "region": r.Region, "country": r.Country, "category": r.Category,
                "tier": r.Tier,
                "held_certs": sorted(r.Held_Certs),
                "required_certs": sorted(r.Required_Certs),
                "missing_certs": r.Missing_Certs,
                "advisory_certs": r.Advisory_Certs,
                "cert_compliant": bool(r.Cert_Compliant),
            },
        }, ensure_ascii=False) + '\n')
print("Saved certifications_rag.jsonl")

106 compliant | 10 missing mandatory certs
Saved certifications_rag.jsonl


## **7. Audit Cadence and Scoring Methadology**

### **7.1. Audit Frequency**

In [59]:
# ============================================================
#  Section 7.1 — Audit Frequency & Overdue Status
# ============================================================
# Cadence: Tier-1 annual, Tier-2 every 6 months, Tier-3 quarterly.
# Overdue thresholds (since Last_Audit_Date):
#   Tier-1 > 14 months, Tier-2 > 7 months, Tier-3 > 4 months  => provisional SWL.
# Active Disruption flag => additional unannounced spot audit within 30 days.
import json

# --- Anchor for "now". Default = dataset's latest audit date (so results are
#     meaningful for this 2023-24 snapshot). Set to pd.Timestamp.today() for live use.
df['Last_Audit'] = pd.to_datetime(df['Last_Audit_Date'])
AS_OF = df['Last_Audit'].max()          # <-- change to pd.Timestamp.today() if desired
print(f"Audit overdue evaluated as of: {AS_OF.date()}")

CADENCE = {'Tier-1': 'Annual', 'Tier-2': 'Bi-annual', 'Tier-3': 'Quarterly'}
OVERDUE_MONTHS = {'Tier-1': 14, 'Tier-2': 7, 'Tier-3': 4}

aud = df.groupby('Supplier_ID').agg(
    Supplier_Name=('Supplier_Name', 'first'), Region=('Region', 'first'),
    Country=('Country', 'first'), Category=('Product_Category', 'first'),
    Tier=('Contract_Tier', 'first'),
    Last_Audit=('Last_Audit', 'max'),                 # most recent audit on record
    Has_Disruption=('Active_Disruptions',
                    lambda s: (s.astype(str).str.lower() != 'none').any()),
).reset_index()

aud['Audit_Cadence'] = aud.Tier.map(CADENCE)
aud['Months_Since_Audit'] = ((AS_OF - aud.Last_Audit).dt.days / 30.44).round(1)
aud['Overdue_Threshold_Months'] = aud.Tier.map(OVERDUE_MONTHS)
aud['Audit_Overdue'] = aud.Months_Since_Audit > aud.Overdue_Threshold_Months
aud['Spot_Audit_Required'] = aud.Has_Disruption          # disruption => 30-day spot audit

aud['_n'] = aud.Supplier_ID.str.replace('SUP-', '', regex=False).astype(int)
aud = aud.sort_values('_n').drop(columns='_n').reset_index(drop=True)

print(f"{int(aud.Audit_Overdue.sum())} overdue | "
      f"{int(aud.Spot_Audit_Required.sum())} require a disruption spot-audit")

# --- RAG JSONL ---
with open('./json_folder/audit_frequency_rag.jsonl', 'w', encoding='utf-8') as f:
    for _, r in aud.iterrows():
        status = ("is AUDIT OVERDUE and placed on provisional Supplier Watch List status"
                  if r.Audit_Overdue else "is within its audit cadence")
        spot = (" An active disruption flag requires an unannounced spot audit within 30 days."
                if r.Spot_Audit_Required else "")
        text = (f"{r.Supplier_Name} ({r.Supplier_ID}), {r.Tier}, {r.Country} ({r.Region}), "
                f"has a {r.Audit_Cadence} audit cadence under Section 7.1. Its last audit was "
                f"{r.Last_Audit.date()} ({r.Months_Since_Audit:.1f} months ago as of "
                f"{AS_OF.date()}); the overdue threshold is {r.Overdue_Threshold_Months} months, "
                f"so the supplier {status}.{spot}")
        f.write(json.dumps({
            "id": f"{r.Supplier_ID}_audit",
            "text": text,
            "metadata": {
                "supplier_id": r.Supplier_ID, "supplier_name": r.Supplier_Name,
                "region": r.Region, "country": r.Country, "category": r.Category,
                "tier": r.Tier, "audit_cadence": r.Audit_Cadence,
                "last_audit_date": str(r.Last_Audit.date()),
                "months_since_audit": float(r.Months_Since_Audit),
                "overdue_threshold_months": int(r.Overdue_Threshold_Months),
                "audit_overdue": bool(r.Audit_Overdue),
                "spot_audit_required": bool(r.Spot_Audit_Required),
                "evaluated_as_of": str(AS_OF.date()),
            },
        }, ensure_ascii=False) + '\n')
print("Saved audit_frequency_rag.jsonl")

Audit overdue evaluated as of: 2024-09-30
1 overdue | 116 require a disruption spot-audit
Saved audit_frequency_rag.jsonl


### **7.2. Audit Score Calculation**

In [60]:
# ============================================================
#  Section 7.2 — Audit Score Calculation & Methodology
# ============================================================
# Compliance Score is a weighted composite (NOTE: sub-components are NOT in the
# dataset, so the score cannot be recomputed — only reported and explained):
#   Quality Systems 30% | Document Compliance 20% | Corrective Action Closure 20%
#   | Regulatory Adherence 15% | Ethical Standards 15%
# Rule: score < 70 at a scheduled audit => mandatory follow-up audit within 60 days
#       (at the supplier's cost).
import json

WEIGHTS = {  # documented for RAG; not used to recompute (components unavailable)
    'Quality Systems': 0.30, 'Document Compliance': 0.20,
    'Corrective Action Closure': 0.20, 'Regulatory Adherence': 0.15,
    'Ethical Standards': 0.15,
}

score = df.groupby('Supplier_ID').agg(
    Supplier_Name=('Supplier_Name', 'first'), Region=('Region', 'first'),
    Country=('Country', 'first'), Category=('Product_Category', 'first'),
    Tier=('Contract_Tier', 'first'),
    Avg_Compliance=('Compliance_Score', 'mean'),
    Min_Compliance=('Compliance_Score', 'min'),
    Max_Compliance=('Compliance_Score', 'max'),
    PO_Count=('PO_ID', 'count'),
).reset_index()

score['Avg_Compliance'] = score.Avg_Compliance.round(2)
score['Followup_Audit_Required'] = score.Min_Compliance < 70   # below 70 at any audit

score['_n'] = score.Supplier_ID.str.replace('SUP-', '', regex=False).astype(int)
score = score.sort_values('_n').drop(columns='_n').reset_index(drop=True)

print(f"{int(score.Followup_Audit_Required.sum())} suppliers require a follow-up audit (<70)")

weight_phrase = ('Quality Systems 30%, Document Compliance 20%, Corrective Action '
                 'Closure 20%, Regulatory Adherence 15%, Ethical Standards 15%')

# --- RAG JSONL ---
with open('./json_folder/audit_scores_rag.jsonl', 'w', encoding='utf-8') as f:
    # one methodology record so "how is the compliance score calculated" retrieves cleanly
    f.write(json.dumps({
        "id": "audit_score_methodology",
        "text": (f"Under Section 7.2, the BQBYTE Compliance Score (0-100) is a weighted "
                 f"composite of five components: {weight_phrase}. A score below 70 at a "
                 f"scheduled audit triggers a mandatory follow-up audit within 60 days at the "
                 f"supplier's cost. The individual component scores are not stored in the "
                 f"supplier performance data, so only the final composite score is available."),
        "metadata": {"scope": "methodology", "section": "7.2",
                     "weights": {k: v for k, v in WEIGHTS.items()}},
    }, ensure_ascii=False) + '\n')

    # per-supplier score records
    for _, r in score.iterrows():
        verdict = ("triggers a mandatory follow-up audit within 60 days at its own cost"
                   if r.Followup_Audit_Required else "is above the 70-point follow-up threshold")
        text = (f"{r.Supplier_Name} ({r.Supplier_ID}), {r.Tier}, {r.Country} ({r.Region}), "
                f"has an average Compliance Score of {r.Avg_Compliance:.1f} across "
                f"{r.PO_Count} audits (range {int(r.Min_Compliance)}-{int(r.Max_Compliance)}). "
                f"The score is the Section 7.2 weighted composite ({weight_phrase}). "
                f"Its lowest recorded score is {int(r.Min_Compliance)}, so the supplier "
                f"{verdict}.")
        f.write(json.dumps({
            "id": f"{r.Supplier_ID}_score",
            "text": text,
            "metadata": {
                "supplier_id": r.Supplier_ID, "supplier_name": r.Supplier_Name,
                "region": r.Region, "country": r.Country, "category": r.Category,
                "tier": r.Tier, "avg_compliance": float(r.Avg_Compliance),
                "min_compliance": int(r.Min_Compliance),
                "max_compliance": int(r.Max_Compliance),
                "followup_audit_required": bool(r.Followup_Audit_Required),
                "po_count": int(r.PO_Count),
            },
        }, ensure_ascii=False) + '\n')
print("Saved audit_scores_rag.jsonl")

69 suppliers require a follow-up audit (<70)
Saved audit_scores_rag.jsonl


## **8. Approved Certification Matrix**

In [61]:
# ============================================================
#  Section 8 — Approved Certification Matrix (policy reference)
# ============================================================
# This is reference content (the master cert table), not a per-supplier
# classification. Each cert becomes one RAG record carrying its definition,
# scope, and how many suppliers in the portfolio currently hold it.
import json

# --- The matrix, transcribed from the policy (page 7) ---
CERT_MATRIX = [
    {"cert": "ISO 9001", "key": "ISO9001",
     "full_name": "Quality Management Systems",
     "mandatory_for": "ALL suppliers, ALL tiers",
     "note": "Non-negotiable baseline",
     "category_scope": ["EC", "RM", "MC", "PM", "IT", "SA"],
     "tier_scope": "all", "conditional": False},
    {"cert": "ISO 14001", "key": "ISO14001",
     "full_name": "Environmental Management",
     "mandatory_for": "Tier-1 RM, PM, EC; all Tier-1 with Sustainability Score requirement",
     "note": "Required for Sustainability Score above 85",
     "category_scope": ["RM", "PM", "EC"],
     "tier_scope": "Tier-1", "conditional": True},
    {"cert": "RoHS", "key": "RoHS",
     "full_name": "Restriction of Hazardous Substances",
     "mandatory_for": "All EC category suppliers",
     "note": "EU directive compliance",
     "category_scope": ["EC"], "tier_scope": "all", "conditional": False},
    {"cert": "REACH", "key": "REACH",
     "full_name": "Registration, Evaluation, Authorisation of Chemicals",
     "mandatory_for": "All RM category (EU-market)",
     "note": "Mandatory for EU-destined Raw Materials",
     "category_scope": ["RM"], "tier_scope": "all", "conditional": True},
    {"cert": "FSC", "key": "FSC",
     "full_name": "Forest Stewardship Council",
     "mandatory_for": "All PM category suppliers",
     "note": "Paper/fibre packaging only",
     "category_scope": ["PM"], "tier_scope": "all", "conditional": False},
    {"cert": "OEKO-TEX", "key": "OEKO-TEX",
     "full_name": "Standard 100 - Textiles",
     "mandatory_for": "All IT category suppliers",
     "note": "Mandatory regardless of tier",
     "category_scope": ["IT"], "tier_scope": "all", "conditional": False},
    {"cert": "AEC-Q100", "key": "AEC-Q100",
     "full_name": "Automotive Electronics Qualification",
     "mandatory_for": "EC suppliers in automotive supply chain",
     "note": "Recommended for Tier-1, mandatory for Tier-2 automotive",
     "category_scope": ["EC"], "tier_scope": "automotive", "conditional": True},
    {"cert": "IATF 16949", "key": "IATF16949",
     "full_name": "Automotive Quality Management System",
     "mandatory_for": "MC suppliers in automotive use",
     "note": "Required for all automotive Mechanical Components",
     "category_scope": ["MC"], "tier_scope": "automotive", "conditional": True},
    {"cert": "AS9100", "key": "AS9100",
     "full_name": "Aerospace Quality Management System",
     "mandatory_for": "Tier-1 SA suppliers (aerospace)",
     "note": "Pairs with NADCAP",
     "category_scope": ["SA"], "tier_scope": "Tier-1 aerospace", "conditional": True},
    {"cert": "NADCAP", "key": "NADCAP",
     "full_name": "Aerospace Special Processes",
     "mandatory_for": "SA suppliers - aerospace applications",
     "note": "Mandatory for aerospace",
     "category_scope": ["SA"], "tier_scope": "aerospace", "conditional": True},
    {"cert": "OSHA", "key": "OSHA",
     "full_name": "Occupational Safety and Health (US)",
     "mandatory_for": "NA-region Tier-1 suppliers",
     "note": "US-specific compliance",
     "category_scope": ["EC", "RM", "MC", "PM", "IT", "SA"],
     "tier_scope": "Tier-1 NA", "conditional": True},
]

# --- Cross-reference: who actually holds each cert ---
def parse_certs(s):
    return set(x.strip() for x in str(s).split(';') if x.strip())

held = df.groupby('Supplier_ID')['Certifications'].apply(
    lambda col: set().union(*[parse_certs(v) for v in col]))
n_suppliers = held.shape[0]

for row in CERT_MATRIX:
    holders = held[held.apply(lambda s: row['key'] in s)].index.tolist()
    row['holder_count'] = len(holders)
    row['holder_ids'] = holders

# --- Tabular export ---
matrix_df = pd.DataFrame([{
    'Cert': r['cert'], 'Full_Name': r['full_name'], 'Mandatory_For': r['mandatory_for'],
    'Category_Scope': ';'.join(r['category_scope']), 'Tier_Scope': r['tier_scope'],
    'Conditional': r['conditional'], 'Holder_Count': r['holder_count'], 'Note': r['note'],
} for r in CERT_MATRIX])

print(matrix_df[['Cert', 'Mandatory_For', 'Holder_Count']].to_string(index=False))

# --- RAG JSONL: one record per certification ---
with open('./json_folder/certification_matrix_rag.jsonl', 'w', encoding='utf-8') as f:
    for r in CERT_MATRIX:
        cond = (" This requirement is conditional (depends on tier, market, or end-use)."
                if r['conditional'] else " This is an unconditional requirement for its scope.")
        text = (f"{r['cert']} ({r['full_name']}) is a certification in the BQBYTE Approved "
                f"Certification Matrix (Section 8). It is mandatory for: {r['mandatory_for']}. "
                f"Note: {r['note']}.{cond} In the current supplier base, {r['holder_count']} of "
                f"{n_suppliers} suppliers hold {r['cert']}.")
        f.write(json.dumps({
            "id": f"cert_matrix_{r['key']}",
            "text": text,
            "metadata": {
                "scope": "certification_reference", "section": "8",
                "cert": r['cert'], "cert_key": r['key'], "full_name": r['full_name'],
                "mandatory_for": r['mandatory_for'], "note": r['note'],
                "category_scope": r['category_scope'], "tier_scope": r['tier_scope'],
                "conditional": r['conditional'], "holder_count": r['holder_count'],
                "holder_ids": r['holder_ids'],
            },
        }, ensure_ascii=False) + '\n')

    # --- a single matrix-summary record for "show me the whole matrix" queries ---
    summary_lines = [f"{r['cert']}: {r['mandatory_for']}" for r in CERT_MATRIX]
    f.write(json.dumps({
        "id": "cert_matrix_summary",
        "text": ("BQBYTE Approved Certification Matrix (Section 8), 11 certifications. "
                 + " | ".join(summary_lines) + ". ISO 9001 is the universal non-negotiable "
                 "baseline for every supplier and tier."),
        "metadata": {"scope": "certification_reference", "section": "8",
                     "cert_count": len(CERT_MATRIX)},
    }, ensure_ascii=False) + '\n')
print("\nSaved certification_matrix_rag.jsonl")

      Cert                                                       Mandatory_For  Holder_Count
  ISO 9001                                            ALL suppliers, ALL tiers           116
 ISO 14001 Tier-1 RM, PM, EC; all Tier-1 with Sustainability Score requirement            22
      RoHS                                           All EC category suppliers            14
     REACH                                         All RM category (EU-market)            11
       FSC                                           All PM category suppliers            21
  OEKO-TEX                                           All IT category suppliers            15
  AEC-Q100                             EC suppliers in automotive supply chain             7
IATF 16949                                      MC suppliers in automotive use            13
    AS9100                                     Tier-1 SA suppliers (aerospace)             4
    NADCAP                               SA suppliers - aerospace appl

## **9. Disruption Response Procedures**

In [62]:
# ============================================================
#  Section 9 — Disruption Response Procedures
# ============================================================
# Response level when a supplier has an active disruption flag:
#   Level 1 - Monitor : Low-risk supplier  -> weekly update, +15% safety stock
#   Level 2 - Manage  : Medium-risk        -> bi-weekly calls, +30% stock, alt on 48h notice
#   Level 3 - Activate: High-risk, OR any mandatory category -> CPO escalation,
#                       alternate activated for >=40% volume in 10 days, +50% stock, 15-day RCA
# Mandatory Level 3 (regardless of risk) — STRICT, policy-named categories only:
#   export control restrictions, active labour strikes, regulatory enforcement,
#   port closure events (>72h).
# Note: the data carries ONE disruption flag per PO, so the "two simultaneous
#       flags" Level-3 branch cannot be evaluated here.
import json

qp = df['PO_Quarter'].str.extract(r'Q(?P<q>\d)-(?P<y>\d{4})')
df['Q'] = qp['q'].astype(int)
df['Y'] = qp['y'].astype(int)

# --- Risk per supplier (latest-quarter snapshot; same logic as Section 5.1) ---
def disruption_factors(text):
    d = str(text).lower(); f = set()
    if 'geopolitical' in d:                                            f.add('geopolitical')
    if any(k in d for k in ['monsoon', 'typhoon', 'flood', 'seasonal']): f.add('seasonal')
    if 'regulatory audit pending' in d or 'compliance review' in d or 'audit overdue' in d:
                                                                       f.add('pending_audit')
    if 'tariff' in d:                                                  f.add('tariff')
    if 'labour' in d or 'strike' in d:                                f.add('labour')
    if 'export' in d:                                                  f.add('export_control')
    if 'currency' in d:                                               f.add('currency')
    if 'port closure' in d:                                            f.add('port_closure')
    return f

def risk_level(text, comp):
    f = disruption_factors(text)
    high = ({'labour', 'export_control', 'currency'} & f) or (comp < 75)
    med = {k for k in ('geopolitical', 'seasonal', 'pending_audit', 'tariff') if k in f}
    if 75 <= comp <= 85:
        med.add('comp_75_85')
    if high or len(med) >= 2:
        return 'High'
    if len(med) >= 1:
        return 'Medium'
    return 'Low'

# STRICT mandatory Level-3 categories (the 4 policy-named ones only)
def is_mandatory_l3(text):
    d = str(text).lower()
    return (('export' in d) or
            ('labour' in d or 'strike' in d) or
            ('enforcement' in d) or
            ('port closure' in d))

comp_avg = df.groupby('Supplier_ID')['Compliance_Score'].mean()

disr = df.sort_values(['Y', 'Q']).groupby('Supplier_ID').agg(
    Supplier_Name=('Supplier_Name', 'first'), Region=('Region', 'first'),
    Country=('Country', 'first'), Category=('Product_Category', 'first'),
    Tier=('Contract_Tier', 'first'),
    Active_Disruption=('Active_Disruptions', 'last'),    # latest-quarter flag
    Quarter=('PO_Quarter', 'last'),
).reset_index()
disr['Comp'] = disr.Supplier_ID.map(comp_avg).round(2)
disr['Risk'] = disr.apply(lambda r: risk_level(r.Active_Disruption, r.Comp), axis=1)
disr['Has_Disruption'] = disr.Active_Disruption.astype(str).str.lower() != 'none'
disr['Mandatory_L3'] = disr.Active_Disruption.apply(is_mandatory_l3)

def response_level(r):
    if not r.Has_Disruption:
        return 'None'
    if r.Mandatory_L3 or r.Risk == 'High':
        return 'Level 3 - Activate'
    if r.Risk == 'Medium':
        return 'Level 2 - Manage'
    return 'Level 1 - Monitor'

# Operational actions per level (for the RAG text + metadata)
ACTIONS = {
    'Level 1 - Monitor':  {'safety_stock_pct': 15, 'cadence': 'weekly status update',
                           'alt_supplier': 'not required'},
    'Level 2 - Manage':   {'safety_stock_pct': 30, 'cadence': 'bi-weekly escalation calls',
                           'alt_supplier': '48-hour readiness notice'},
    'Level 3 - Activate': {'safety_stock_pct': 50, 'cadence': 'immediate CPO escalation',
                           'alt_supplier': 'activate >=40% volume within 10 business days'},
    'None':               {'safety_stock_pct': 0, 'cadence': 'n/a', 'alt_supplier': 'n/a'},
}

disr['Response_Level'] = disr.apply(response_level, axis=1)
disr['Safety_Stock_Adj_Pct'] = disr.Response_Level.map(lambda l: ACTIONS[l]['safety_stock_pct'])

disr['_n'] = disr.Supplier_ID.str.replace('SUP-', '', regex=False).astype(int)
disr = disr.sort_values('_n').drop(columns='_n').reset_index(drop=True)

print(disr.Response_Level.value_counts().to_dict())

# --- RAG JSONL (only suppliers with an active disruption are actionable) ---
with open('./json_folder/disruption_response_rag.jsonl', 'w', encoding='utf-8') as f:
    for _, r in disr[disr.Has_Disruption].iterrows():
        a = ACTIONS[r.Response_Level]
        why = ("a mandatory Level-3 disruption category (export control, labour strike, "
               "regulatory enforcement, or port closure)" if r.Mandatory_L3
               else f"its {r.Risk} risk level")
        text = (f"{r.Supplier_Name} ({r.Supplier_ID}), {r.Tier}, {r.Country} ({r.Region}), "
                f"has an active disruption '{r.Active_Disruption}' as of {r.Quarter}. Under "
                f"Section 9 this warrants a {r.Response_Level} response, driven by {why}. "
                f"Required actions: {a['cadence']}, safety stock adjusted by "
                f"+{a['safety_stock_pct']}%, alternate supplier {a['alt_supplier']}.")
        f.write(json.dumps({
            "id": f"{r.Supplier_ID}_disruption",
            "text": text,
            "metadata": {
                "supplier_id": r.Supplier_ID, "supplier_name": r.Supplier_Name,
                "region": r.Region, "country": r.Country, "category": r.Category,
                "tier": r.Tier, "quarter": r.Quarter,
                "active_disruption": str(r.Active_Disruption),
                "risk_level": r.Risk, "response_level": r.Response_Level,
                "mandatory_l3": bool(r.Mandatory_L3),
                "safety_stock_adj_pct": int(a['safety_stock_pct']),
                "alt_supplier_action": a['alt_supplier'],
            },
        }, ensure_ascii=False) + '\n')

    # --- reference record describing the three levels themselves ---
    f.write(json.dumps({
        "id": "disruption_response_levels_reference",
        "text": ("Section 9 defines three disruption response levels. Level 1 (Monitor): "
                 "Low-risk suppliers with a disruption flag - weekly status updates, safety "
                 "stock +15%. Level 2 (Manage): Medium-risk - bi-weekly escalation calls, "
                 "safety stock +30%, alternate supplier on 48-hour readiness notice. Level 3 "
                 "(Activate): High-risk or any mandatory category - immediate CPO escalation, "
                 "alternate activated for at least 40% of volume within 10 business days, "
                 "safety stock +50%, full RCA within 15 business days. Mandatory Level 3 "
                 "categories regardless of risk: export control restrictions, active labour "
                 "strikes, regulatory enforcement actions, and port closures exceeding 72 hours."),
        "metadata": {"scope": "disruption_reference", "section": "9"},
    }, ensure_ascii=False) + '\n')
print("Saved disruption_response_rag.jsonl")

{'Level 3 - Activate': 51, 'Level 2 - Manage': 33, 'Level 1 - Monitor': 32}
Saved disruption_response_rag.jsonl


## **10.Alternate Supplier Activation Rules**

In [63]:
# ============================================================
#  Section 10 — Alternate Supplier Activation Rules
# ============================================================
# Every Tier-2/3 supplier MUST have a designated alternate (Tier-1 recommended).
# An alternate must satisfy four conditions:
#   (a) lead time within 120% of the primary's quoted lead time
#   (b) hold all mandatory certifications for the primary's category (Section 6.2)
#   (c) NOT share the primary's geopolitical region — applies only in a Level-3
#       activation scenario (so flagged only for suppliers currently at Level 3)
#   (d) NOT be on SWL status (compliance < 60 at any audit) at activation
# Activation volume: 40%-100% of primary volume per disruption severity / CPO directive.
import json

qp = df['PO_Quarter'].str.extract(r'Q(?P<q>\d)-(?P<y>\d{4})')
df['Q'] = qp['q'].astype(int)
df['Y'] = qp['y'].astype(int)

# --- 1. Identify the current Level-3 supplier set (reuse Sections 5 & 9 logic) ---
def disruption_factors(t):
    d = str(t).lower(); f = set()
    if 'geopolitical' in d:                                            f.add('geopolitical')
    if any(k in d for k in ['monsoon', 'typhoon', 'flood', 'seasonal']): f.add('seasonal')
    if 'regulatory audit pending' in d or 'compliance review' in d or 'audit overdue' in d:
                                                                       f.add('pending_audit')
    if 'tariff' in d:                                                  f.add('tariff')
    if 'labour' in d or 'strike' in d:                                f.add('labour')
    if 'export' in d:                                                  f.add('export_control')
    if 'currency' in d:                                               f.add('currency')
    if 'port closure' in d:                                            f.add('port_closure')
    return f

def risk_level(t, c):
    f = disruption_factors(t)
    high = ({'labour', 'export_control', 'currency'} & f) or (c < 75)
    med = {k for k in ('geopolitical', 'seasonal', 'pending_audit', 'tariff') if k in f}
    if 75 <= c <= 85:
        med.add('comp_75_85')
    return 'High' if (high or len(med) >= 2) else ('Medium' if med else 'Low')

def mandatory_l3(t):
    d = str(t).lower()
    return ('export' in d) or ('labour' in d or 'strike' in d) or \
           ('enforcement' in d) or ('port closure' in d)

comp_avg = df.groupby('Supplier_ID')['Compliance_Score'].mean()
latest = df.sort_values(['Y', 'Q']).groupby('Supplier_ID').agg(
    Disr=('Active_Disruptions', 'last')).reset_index()
latest['Comp'] = latest.Supplier_ID.map(comp_avg)
latest['L3'] = latest.apply(
    lambda r: (mandatory_l3(r.Disr) or risk_level(r.Disr, r.Comp) == 'High')
              and str(r.Disr).lower() != 'none', axis=1)
L3_SET = set(latest[latest.L3].Supplier_ID)

# --- 2. Supplier reference table + Section 6.2 cert requirements ---
def parse_certs(s):
    return set(x.strip() for x in str(s).split(';') if x.strip())

ref = df.groupby('Supplier_ID').agg(
    Name=('Supplier_Name', 'first'), Region=('Region', 'first'),
    Country=('Country', 'first'), Cat=('Product_Category', 'first'),
    Tier=('Contract_Tier', 'first'), Lead=('Lead_Time_Days', 'mean'),
    MinComp=('Compliance_Score', 'min'), Alt=('Alt_Supplier_ID', 'first'),
).reset_index()
ref['Certs'] = df.groupby('Supplier_ID')['Certifications'].apply(
    lambda c: set().union(*[parse_certs(v) for v in c])).values
ref = ref.set_index('Supplier_ID')

CAT = {'Electronic Components': 'EC', 'Raw Materials': 'RM',
       'Mechanical Components': 'MC', 'Packaging Materials': 'PM',
       'Industrial Textiles': 'IT', 'Specialty Alloys': 'SA'}

def required_certs(tier, cat_full, region):
    cat = CAT[cat_full]; req = {'ISO9001'}
    if cat == 'EC': req |= {'RoHS'}
    if cat == 'RM':
        if region == 'EMEA': req |= {'REACH'}
        if tier == 'Tier-1': req |= {'ISO14001'}
    if cat == 'PM':
        req |= {'FSC'}
        if tier == 'Tier-1': req |= {'ISO14001'}
    if cat == 'IT': req |= {'OEKO-TEX'}
    if cat == 'SA' and tier == 'Tier-1': req |= {'AS9100'}
    return req

# --- 3. Evaluate each supplier's alternate against the four conditions ---
def evaluate_alt(sid):
    r = ref.loc[sid]; alt = r['Alt']
    if pd.isna(alt) or alt not in ref.index:
        return {'alt': alt, 'eligible': False, 'issues': ['alt_not_in_supplier_set']}
    a = ref.loc[alt]
    issues = []
    if a['Lead'] > 1.2 * r['Lead']:
        issues.append('lead_time_exceeds_120pct')
    needed = required_certs(r['Tier'], r['Cat'], a['Region'])   # alt must meet primary-category reqs
    missing = sorted(needed - a['Certs'])
    if missing:
        issues.append('missing_certs:' + '+'.join(missing))
    if sid in L3_SET and a['Region'] == r['Region']:
        issues.append('same_region_in_L3_scenario')
    if a['MinComp'] < 60:
        issues.append('alternate_on_SWL')
    return {'alt': alt, 'alt_name': a['Name'], 'alt_region': a['Region'],
            'eligible': len(issues) == 0, 'issues': issues}

rows = []
for sid in ref.index:
    r = ref.loc[sid]
    ev = evaluate_alt(sid)
    has_alt = pd.notna(r['Alt'])
    alt_mandatory = r['Tier'] in ('Tier-2', 'Tier-3')
    rows.append({
        'Supplier_ID': sid, 'Supplier_Name': r['Name'], 'Region': r['Region'],
        'Country': r['Country'], 'Category': r['Cat'], 'Tier': r['Tier'],
        'Alt_Supplier_ID': r['Alt'], 'Alt_Mandatory': alt_mandatory,
        'Has_Alt': has_alt, 'In_L3_Scenario': sid in L3_SET,
        'Alt_Eligible': ev['eligible'], 'Alt_Issues': ev['issues'],
        'Alt_Region': ev.get('alt_region'), 'Alt_Name': ev.get('alt_name'),
    })

alt_df = pd.DataFrame(rows)
alt_df['_n'] = alt_df.Supplier_ID.str.replace('SUP-', '', regex=False).astype(int)
alt_df = alt_df.sort_values('_n').drop(columns='_n').reset_index(drop=True)

alt_csv = alt_df.copy()
alt_csv['Alt_Issues'] = alt_csv.Alt_Issues.apply(lambda l: ';'.join(l))


print(f"Eligible alternates: {int(alt_df.Alt_Eligible.sum())} | "
      f"with issues: {int((~alt_df.Alt_Eligible).sum())}")
print(f"Tier-2/3 missing a mandatory alternate: "
      f"{int(((alt_df.Alt_Mandatory) & (~alt_df.Has_Alt)).sum())}")

# --- 4. RAG JSONL ---
ISSUE_LABEL = {
    'lead_time_exceeds_120pct': "the alternate's lead time exceeds 120% of the primary's",
    'same_region_in_L3_scenario': "the alternate shares the primary's region, disallowed in a Level-3 activation",
    'alternate_on_SWL': "the alternate is on Supplier Watch List status",
    'alt_not_in_supplier_set': "no valid alternate is designated in the supplier base",
}
def describe_issue(i):
    if i.startswith('missing_certs:'):
        return f"the alternate is missing mandatory certifications ({i.split(':',1)[1].replace('+', ', ')})"
    return ISSUE_LABEL.get(i, i)

with open('./json_folder/alternate_suppliers_rag.jsonl', 'w', encoding='utf-8') as f:
    for _, r in alt_df.iterrows():
        if not r.Has_Alt:
            verdict = ("has NO designated alternate supplier" +
                       (" — a violation, since alternates are mandatory for this tier"
                        if r.Alt_Mandatory else " (recommended but not mandatory for Tier-1)"))
        elif r.Alt_Eligible:
            verdict = (f"has alternate {r.Alt_Supplier_ID} ({r.Alt_Name}, {r.Alt_Region}), "
                       f"which satisfies all Section 10 activation conditions")
        else:
            reasons = "; ".join(describe_issue(i) for i in r.Alt_Issues)
            verdict = (f"has alternate {r.Alt_Supplier_ID} ({r.Alt_Name}, {r.Alt_Region}), "
                       f"which is NOT currently eligible because {reasons}")
        l3 = " The primary is currently in a Level-3 disruption scenario." if r.In_L3_Scenario else ""
        text = (f"{r.Supplier_Name} ({r.Supplier_ID}), {r.Tier}, {r.Country} ({r.Region}), "
                f"supplying {r.Category}, {verdict}. Under Section 10, activation allocates "
                f"40%-100% of primary volume per disruption severity and CPO directive.{l3}")
        f.write(json.dumps({
            "id": f"{r.Supplier_ID}_altsupplier",
            "text": text,
            "metadata": {
                "supplier_id": r.Supplier_ID, "supplier_name": r.Supplier_Name,
                "region": r.Region, "country": r.Country, "category": r.Category,
                "tier": r.Tier, "alt_supplier_id": (r.Alt_Supplier_ID if r.Has_Alt else None),
                "alt_region": r.Alt_Region, "alt_mandatory": bool(r.Alt_Mandatory),
                "has_alt": bool(r.Has_Alt), "alt_eligible": bool(r.Alt_Eligible),
                "alt_issues": r.Alt_Issues, "in_l3_scenario": bool(r.In_L3_Scenario),
            },
        }, ensure_ascii=False) + '\n')

    # --- discrepancy note: the policy's SUP-009/017/003 example doesn't match the data ---
    f.write(json.dumps({
        "id": "alt_supplier_policy_note_discrepancy",
        "text": ("Section 10 includes a note stating suppliers SUP-009 and SUP-017 (NA, Raw "
                 "Materials) share alternate SUP-003 (EMEA, Germany). In the supplier "
                 "performance dataset this does not hold: SUP-009 and SUP-017 are APAC "
                 "Specialty Alloys suppliers with alternates SUP-106 and SUP-114 respectively, "
                 "and SUP-003 is an APAC Raw Materials supplier. The policy example appears "
                 "illustrative rather than reflective of the current data."),
        "metadata": {"scope": "alt_supplier_reference", "section": "10",
                     "note_type": "policy_data_discrepancy"},
    }, ensure_ascii=False) + '\n')
print("Saved alternate_suppliers_rag.jsonl")

Eligible alternates: 28 | with issues: 88
Tier-2/3 missing a mandatory alternate: 0
Saved alternate_suppliers_rag.jsonl
